# LAB | Week 3 | Day 1 | From hacky script to "we could ship this"
## John Adams

**Description:**
Taking a previous lab notebook and refactoring with AI workflow path. Shows data moves from input to output correctly.

### Step 1: Select Your Code

**Original notebook copy:** `product_listing_generator_original.ipynb`

**This labs notebook:** `product_listing_generator_refactored.ipynb` is the working file where all refactoring changes are made.

### Step 2: Analyze Your Code

#### Refactoring Checklist

**Issues Found:**
- [ ] Low-resolution product images cause the model to silently ignore the image and generate a listing from text metadata alone, with no error or warning (silent failure)
- [ ] Function `generate_product_listing` does too much (builds prompt AND encodes image AND calls API AND parses JSON response)
- [ ] JSON parse failure is caught but not clearly surfaced with function name/location context
- [ ] Error handling missing in `encode_image_to_base64` (network request can fail with no context)
- [ ] Error handling missing around price parsing (malformed price string crashes with no context)
- [ ] Code repeated in price-cleaning logic (appears in both the prompt-test cell and the batch-processing loop) (could be helper function)
- [ ] Hardcoded values in `generate_product_listing` and `encode_image_to_base64` (model name, temperature, max_tokens, min_size)
- [ ] Pydantic imported but unused — no schema validates the parsed listing's shape

**Priority:**
1. Image-grounding silent failure (defeats the core objective of the lab: listings generated *from the image*)
2. `generate_product_listing` monolithic function (root cause that makes the other issues hard to fix independently)
3. Missing/unclear error handling (JSON parsing, network calls, price parsing)


### Step 3: Create Helper Functions

Helper for the repeated price-cleaning code found in Step 2.


In [64]:
# Clean a price string like '$14.47' into a float
def parse_price(price_str):
    return float(price_str.replace("$", "").replace(",", ""))


In [65]:
# Test parse_price independently
assert parse_price("$14.47") == 14.47
assert parse_price("$1,234.56") == 1234.56
assert parse_price("9.06") == 9.06
print("parse_price: all tests passed")


parse_price: all tests passed


Helper for cleaning the model's raw JSON text before parsing, found in Step 2.


In [66]:
# Strip markdown code fences from the model's raw text before parsing as JSON
def clean_json_response(raw_content):
    return raw_content.strip().strip("`").replace("json\n", "", 1).strip()


In [67]:
# Test clean_json_response independently
assert clean_json_response('```json\n{"a": 1}\n```') == '{"a": 1}'
assert clean_json_response('{"a": 1}') == '{"a": 1}'
print("clean_json_response: all tests passed")


clean_json_response: all tests passed


Helper for the top-priority issue from Step 2: flag images too small for the model to reliably see, since that caused silent, ungrounded listings before.


In [68]:
# Flag whether an image is below the minimum size the model can reliably see
def check_image_resolution(image, min_size=200):
    width, height = image.size
    return min(width, height) >= min_size


In [69]:
# Test check_image_resolution independently
from PIL import Image
small_img = Image.new("RGB", (60, 80))
big_img = Image.new("RGB", (500, 500))
assert check_image_resolution(small_img) == False
assert check_image_resolution(big_img) == True
print("check_image_resolution: all tests passed")


check_image_resolution: all tests passed


Pulling `encode_image_to_base64` out as its own tested helper, separate from the API-calling logic that used it inline.


In [70]:
# Download an image from a URL and encode it to base64, upscaling if too small
import io, base64, requests
from PIL import Image

def encode_image_to_base64(image_url, upscale_target=512):
    try:
        response = requests.get(image_url, timeout=10)
        response.raise_for_status()
    except requests.exceptions.Timeout:
        raise RuntimeError(f"encode_image_to_base64: timed out downloading image from {image_url}. Check your network connection and retry.")
    except requests.exceptions.ConnectionError:
        raise RuntimeError(f"encode_image_to_base64: could not connect to {image_url}. Check the URL and your network connection.")
    except requests.exceptions.HTTPError as e:
        raise RuntimeError(f"encode_image_to_base64: server returned an error for {image_url}: {e}. Check the image URL is valid.")

    image = Image.open(io.BytesIO(response.content)).convert("RGB")
    is_high_res = check_image_resolution(image)

    if min(image.size) < upscale_target:
        scale = upscale_target / min(image.size)
        new_size = (int(image.width * scale), int(image.height * scale))
        image = image.resize(new_size, Image.LANCZOS)

    buffer = io.BytesIO()
    image.save(buffer, format="JPEG")
    img_str = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return img_str, is_high_res


In [71]:
# Test encode_image_to_base64 independently, using a real product image URL
sample_url = "https://images-na.ssl-images-amazon.com/images/I/51FUt3TYecL.jpg"
encoded, was_high_res = encode_image_to_base64(sample_url)
assert isinstance(encoded, str) and len(encoded) > 0
assert isinstance(was_high_res, bool)
print(f"encode_image_to_base64: test passed, was_high_res={was_high_res}")


encode_image_to_base64: test passed, was_high_res=True


### Step 4: Modularize Your Functions


Splitting `generate_product_listing` (Step 2 issue: builds prompt AND encodes image AND calls API AND parses response) into 3 single-responsibility functions, composed by a slim main function.


Client setup, self-contained here so Step 4's functions don't depend on the old aux code running first.


In [72]:
# Load API key and create the OpenAI client
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


Moved in from the original code, unchanged, so Step 4's functions are self-contained.


In [73]:
# Build the text prompt asking the model for a structured product listing
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}
 
Please create a professional product listing that includes:
 
1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)
 
Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", ...],
    "keywords": "keyword1, keyword2, ..."
}}
 
Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features."""
    return prompt


Piece 1 of the Step 4 split: builds the prompt and encodes the image into the messages format the API expects.


In [74]:
# Build the prompt text and encode the image into one messages list for the API call
def build_vision_messages(product_name, price, category, image_url):
    prompt = create_product_listing_prompt(product_name, price, category)
    encoded_image, is_high_res = encode_image_to_base64(image_url)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{encoded_image}",
                        "detail": "high"
                    }
                }
            ]
        }
    ]
    return messages, is_high_res


Piece 2 of the Step 4 split: the only place an OpenAI API call happens, so it's the only place an OpenAI API error can occur.


In [75]:
# Call the OpenAI vision API with the given messages
from openai import APIError, APIConnectionError, RateLimitError

def call_vision_api(messages, model="gpt-4o-mini", temperature=0.7, max_tokens=1000):
    try:
        response = client.chat.completions.create(
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            messages=messages
        )
    except RateLimitError as e:
        raise RuntimeError(f"call_vision_api: rate limit hit calling {model}: {e}. Wait and retry, or check your API plan limits.")
    except APIConnectionError as e:
        raise RuntimeError(f"call_vision_api: network error connecting to OpenAI: {e}. Check your internet connection and retry.")
    except APIError as e:
        raise RuntimeError(f"call_vision_api: OpenAI API error calling {model}: {e}. Check your API key and request parameters.")

    return response.choices[0].message.content


Piece 3 of the Step 4 split: cleans and JSON-parses the model's raw response.


In [76]:
# Clean and JSON-parse the model's raw text response into a listing dict
import json

def parse_listing_response(raw_content):
    cleaned = clean_json_response(raw_content)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        print(f"parse_listing_response: JSON parsing failed at line {e.lineno}, col {e.colno}: {e.msg}")
        return {"raw_response": raw_content}


Composes the 3 pieces above into the main listing function. This version is superseded by the Step 5 version below, which adds Pydantic validation.


In [77]:
# Compose the 3 pieces: build messages, call the API, parse the response
def generate_product_listing(product_name, price, category, image_url, label="Product"):
    messages, is_high_res = build_vision_messages(product_name, price, category, image_url)
    raw_content = call_vision_api(messages)
    listing = parse_listing_response(raw_content)

    listing["is_high_res"] = is_high_res
    print(f"{label}\n{raw_content}\n")
    return listing


Testing the Step 4 version of `generate_product_listing`, before Step 5 adds validation.


In [78]:
# Test generate_product_listing on one real product
sample_url = "https://images-na.ssl-images-amazon.com/images/I/51FUt3TYecL.jpg"
listing = generate_product_listing(
    product_name="Craft-tastic Empower Poster",
    price=14.47,
    category="Toys & Games",
    image_url=sample_url,
    label="Step 4 smoke test",
)
assert "is_high_res" in listing
print("generate_product_listing: test passed")


Step 4 smoke test
```json
{
    "title": "Craft-tastic Empower Poster - Inspire & Motivate",
    "description": "Unleash creativity and boost self-esteem with the Craft-tastic Empower Poster! This vibrant and colorful poster features a dynamic burst of affirmations, such as 'I am Brave,' 'I am Unique,' and 'I am Happy,' designed to inspire both kids and adults alike. Measuring 9.75\" x 13.75\", it’s perfect for decorating a bedroom, playroom, or classroom. Made with high-quality materials, the poster not only looks great but is also durable enough to withstand the test of time. Encourage positive thinking and self-empowerment with this eye-catching decor piece that serves as a daily reminder of one’s strengths and potential. Whether it’s a gift for a friend or a treat for yourself, the Craft-tastic Empower Poster is a wonderful addition to any space, fostering an environment of positivity and inspiration.",
    "features": [
        "Vibrant and colorful design",
        "Inspiring aff

### Step 5: Add Error Handling


Updated `encode_image_to_base64` (network errors) and `call_vision_api` (OpenAI API errors) above, in place, so Step 4's functions carry error handling from the start. Adding Pydantic validation here, since that resolves the unused-import issue from Step 2 and covers the last required error type.


Defines the shape a valid listing must have, so a malformed model response is caught explicitly instead of used as-is.


In [79]:
# Schema a valid product listing must match
from pydantic import BaseModel, ValidationError

class ProductListing(BaseModel):
    title: str
    description: str
    features: list[str]
    keywords: str


Uses the schema above to validate a parsed listing dict, raising a clear error if the model's response doesn't match.


In [80]:
# Validate a parsed listing dict against the ProductListing schema
def validate_listing(listing_dict):
    try:
        return ProductListing(**listing_dict)
    except ValidationError as e:
        raise ValueError(f"validate_listing: model response did not match the expected listing shape: {e}. Check the prompt still requests title, description, features, and keywords.")


In [81]:
# Test validate_listing independently, valid and invalid cases
good = {"title": "A", "description": "B", "features": ["x"], "keywords": "y"}
assert validate_listing(good).title == "A"

bad = {"title": "A"}
try:
    validate_listing(bad)
    raise AssertionError("expected ValueError for missing fields")
except ValueError as e:
    print(f"Caught expected error: {e}")

print("validate_listing: all tests passed")


Caught expected error: validate_listing: model response did not match the expected listing shape: 3 validation errors for ProductListing
description
  Field required [type=missing, input_value={'title': 'A'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
features
  Field required [type=missing, input_value={'title': 'A'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
keywords
  Field required [type=missing, input_value={'title': 'A'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing. Check the prompt still requests title, description, features, and keywords.
validate_listing: all tests passed


Replaces the Step 4 version of `generate_product_listing` above: same 3-piece composition, now also validates the parsed listing against the `ProductListing` schema before returning it.


In [82]:
# Compose the pieces: build messages, call the API, parse and validate the response
def generate_product_listing(product_name, price, category, image_url, label="Product"):
    messages, is_high_res = build_vision_messages(product_name, price, category, image_url)
    raw_content = call_vision_api(messages)
    listing_dict = parse_listing_response(raw_content)

    if "raw_response" in listing_dict:
        listing_dict["is_high_res"] = is_high_res
        print(f"{label}\n{raw_content}\n")
        return listing_dict

    validated = validate_listing(listing_dict)
    listing = validated.model_dump()
    listing["is_high_res"] = is_high_res

    print(f"{label}\n{raw_content}\n")
    return listing


In [83]:
# Test the updated generate_product_listing on one real product
sample_url = "https://images-na.ssl-images-amazon.com/images/I/51FUt3TYecL.jpg"
listing = generate_product_listing(
    product_name="Craft-tastic Empower Poster",
    price=14.47,
    category="Toys & Games",
    image_url=sample_url,
    label="Step 5 smoke test",
)
assert "is_high_res" in listing
assert "title" in listing
print("generate_product_listing (with validation): test passed")


Step 5 smoke test
```json
{
    "title": "Craft-tastic Empower Poster - Inspire & Create",
    "description": "Unleash creativity and boost self-esteem with the Craft-tastic Empower Poster! This vibrant, eye-catching poster measures 9.75\" x 13.75\" and features a dazzling array of colorful rays radiating from a central 'I am' emblem. Each ray is adorned with uplifting words like 'Brave', 'Unique', 'Happy', and 'Courageous', designed to inspire kids and adults alike. Perfect for bedrooms, playrooms, or classrooms, this poster serves as a daily reminder of one's strengths and potential. Crafted from high-quality materials, it’s durable and easy to hang, making it a fantastic addition to any space. Encourage positivity and self-affirmation while adding a splash of color to your décor. Ideal for gifting or personal use, the Craft-tastic Empower Poster is more than just decor; it's an empowering statement that celebrates individuality and creativity.",
    "features": [
        "Vibrant, m

Confirming the network error path shows WHERE it failed, using a bad URL.


In [84]:
# Test encode_image_to_base64 error handling with an unreachable URL
try:
    encode_image_to_base64("https://this-domain-does-not-exist-12345.example/image.jpg")
    raise AssertionError("expected a RuntimeError for an unreachable URL")
except RuntimeError as e:
    print(f"Caught expected error: {e}")


Caught expected error: encode_image_to_base64: could not connect to https://this-domain-does-not-exist-12345.example/image.jpg. Check the URL and your network connection.


Confirming the OpenAI API error path shows WHERE it failed, using a deliberately invalid key on a separate client so the real `client` is untouched.


In [85]:
# Test call_vision_api error handling with a deliberately invalid API key
from openai import OpenAI as _OpenAI

bad_client = _OpenAI(api_key="sk-invalid-key-for-testing")
real_client = client
client = bad_client

try:
    call_vision_api([{"role": "user", "content": "test"}])
    raise AssertionError("expected a RuntimeError for an invalid API key")
except RuntimeError as e:
    print(f"Caught expected error: {e}")
finally:
    client = real_client


Caught expected error: call_vision_api: OpenAI API error calling gpt-4o-mini: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-inval**************ting. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}. Check your API key and request parameters.


Loads a saved listings JSON file, the last required error type: `FileNotFoundError`, shown with the path and a suggestion.


In [86]:
# Load a saved listings JSON file from disk
def load_listings_file(path):
    try:
        with open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        raise FileNotFoundError(f"load_listings_file: no file found at '{path}'. Check the path is correct and the file has been generated.")


In [87]:
# Test load_listings_file, valid path and missing path
listings = load_listings_file("generated_listings.json")
assert isinstance(listings, list) and len(listings) > 0
print(f"Loaded {len(listings)} listings")

try:
    load_listings_file("does_not_exist.json")
    raise AssertionError("expected a FileNotFoundError for a missing file")
except FileNotFoundError as e:
    print(f"Caught expected error: {e}")


Loaded 5 listings
Caught expected error: load_listings_file: no file found at 'does_not_exist.json'. Check the path is correct and the file has been generated.


### Step 6: Test Your Refactored Code


Loads the same 5 live products the original used (`cvnberk/amazon-products`, same deterministic slice), runs them through the refactored `generate_product_listing`, and saves the results for a direct comparison against `generated_listings.json` (the original's output).


Load the same 5 products the original code used.


In [88]:
# Load the same 5-product dataset slice the original code used
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("cvnberk/amazon-products", split="train[:5]")
products_df = pd.DataFrame(dataset)
print(f"Loaded {len(products_df)} products")


Loaded 5 products


Run the refactored pipeline on live data for all 5 products, same as the original's batch loop.


In [89]:
# Run the refactored generate_product_listing on all 5 live products
refactored_results = []

for idx, row in products_df.iterrows():
    print(f"Processing product {idx}: {row['Product Name']}")
    try:
        price = parse_price(row["Selling Price"])
        listing = generate_product_listing(
            product_name=row["Product Name"],
            price=price,
            category=row["Category"],
            image_url=row["Image"],
            label=f"Product listing {idx + 1} (row id {idx})",
        )
        refactored_results.append({"id": idx, "listing": listing, "error": None})
    except (RuntimeError, ValueError) as e:
        print(f"Failed: {e}")
        refactored_results.append({"id": idx, "listing": None, "error": str(e)})


Processing product 0: Craft-tastic – Empower Poster – Craft Kit – Design a One-of-a-Kind Inspirational Poster
Product listing 1 (row id 0)
```json
{
    "title": "Craft-tastic Empower Poster Craft Kit - Inspire & Create",
    "description": "Unleash your creativity with the Craft-tastic Empower Poster Craft Kit! This vibrant DIY kit allows you to design a unique, inspirational poster that not only decorates your space but also uplifts your spirit. Featuring a colorful burst of radiant hues, including pinks, blues, and greens, the poster showcases empowering words like 'Courageous,' 'Joyful,' and 'Unique.' Measuring 9.75\" x 13.75,\" this poster provides ample space for personal expression. Perfect for all ages, this kit includes everything you need to create your masterpiece – from pre-cut letters to a sturdy backing. Whether it’s for a bedroom, classroom, or creative gift, this craft kit encourages positivity and self-reflection. Dive into a world of color and inspiration while enhanc

Save the refactored pipeline's output for comparison against the original's `generated_listings.json`.


In [90]:
# Save the refactored pipeline's results to its own file
with open("generated_listings_refactored.json", "w") as f:
    json.dump(refactored_results, f, indent=2)

succeeded = sum(1 for r in refactored_results if r["error"] is None)
failed = sum(1 for r in refactored_results if r["error"] is not None)
print(f"BATCH COMPLETE: {succeeded}/{len(refactored_results)} succeeded, {failed} failed")
print("Saved to generated_listings_refactored.json")


BATCH COMPLETE: 5/5 succeeded, 0 failed
Saved to generated_listings_refactored.json
